# Problem: Payroll Calculation Engine
**Difficulty:** Easy  
**Company:** Odoo  

## Description
You are a data analyst on the payroll team at Odoo. The weekly payroll run needs each employee's gross pay computed from their logged hours, applying the standard overtime policy before checks go out.

Compute each employee's gross pay from their logged hours based on these business rules:
- The first **40 hours** are paid at the employee's standard base hourly rate.
- Hours **strictly above 40** are paid at **1.5 times** the hourly rate.
- Working **exactly 40 hours** (or fewer) earns no overtime premium.
- Round gross pay to **2 decimal places**.
- Return only employees included in the current payroll run (`INNER JOIN`).
- Output columns: `employee_id`, `name`, `pay`, `position`
- Sort results by `employee_id` in **ascending order**.

---

## Schema

### `rp_employees`
| Column | Type | Description |
|---|---|---|
| `employee_id` | Integer | Unique identifier for the employee |
| `name` | String | Employee full name |
| `age` | Integer | Employee age |
| `position` | String | Job title/role |

### `rp_payroll`
| Column | Type | Description |
|---|---|---|
| `employee_id` | Integer | Identifier mapping to `rp_employees` |
| `hours_worked` | Double / Float | Total logged hours in the week |
| `hourly_rate` | Double / Float | Standard base pay rate per hour |

---

## Example 1

### Input
**rp_employees:**
| employee_id | name | age | position |
|---|---|---|---|
| 1 | Alice | 25 | Software Engineer |
| 2 | Bob | 30 | Data Analyst |
| 3 | Carol | 28 | Product Manager |
| 4 | Dave | 24 | Software Engineer |
| 5 | Eve | 31 | QA Engineer |

**rp_payroll:**
| employee_id | hours_worked | hourly_rate |
|---|---|---|
| 1 | 45.0 | 30.0 |
| 2 | 38.0 | 25.0 |
| 3 | 41.5 | 35.0 |
| 4 | 40.0 | 28.0 |

### Expected Output
| employee_id | name | pay | position |
|---|---|---|---|
| 1 | Alice | 1425.00 | Software Engineer |
| 2 | Bob | 950.00 | Data Analyst |
| 3 | Carol | 1478.75 | Product Manager |
| 4 | Dave | 1120.00 | Software Engineer |

### Explanation
- **Alice (45 hrs @ $30/hr):** `40 * 30 = 1200` + `5 OT hrs * (1.5 * 30) = 225` $\rightarrow$ **1425.00**
- **Bob (38 hrs @ $25/hr):** Under 40 hrs $\rightarrow$ `38 * 25` = **950.00**
- **Carol (41.5 hrs @ $35/hr):** `40 * 35 = 1400` + `1.5 OT hrs * (1.5 * 35) = 78.75` $\rightarrow$ **1478.75**
- **Dave (40 hrs @ $28/hr):** Exactly 40 hrs (no OT) $\rightarrow$ `40 * 28` = **1120.00**
- **Eve:** Not in `rp_payroll`, excluded from the run.

In [1]:
# 1. Install pyspark (if not already installed in Colab runtime)
!pip install -q pyspark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, DoubleType
)

In [2]:
# 2. Initialize Spark Session
spark = SparkSession.builder \
    .appName("PayrollCalculationEngine") \
    .getOrCreate()

In [3]:
# ==============================================================================
# 3. SCHEMA & SAMPLE DATA
# ==============================================================================
emp_schema = StructType([
    StructField("employee_id", IntegerType(), False),
    StructField("name", StringType(), False),
    StructField("age", IntegerType(), True),
    StructField("position", StringType(), True)
])

payroll_schema = StructType([
    StructField("employee_id", IntegerType(), False),
    StructField("hours_worked", DoubleType(), False),
    StructField("hourly_rate", DoubleType(), False)
])

# Example 1 Data
emp_data = [
    (1, "Alice", 25, "Software Engineer"),
    (2, "Bob", 30, "Data Analyst"),
    (3, "Carol", 28, "Product Manager"),
    (4, "Dave", 24, "Software Engineer"),
    (5, "Eve", 31, "QA Engineer")
]

payroll_data = [
    (1, 45.0, 30.0),
    (2, 38.0, 25.0),
    (3, 41.5, 35.0),
    (4, 40.0, 28.0)
]

df_employees = spark.createDataFrame(emp_data, schema=emp_schema)
df_payroll = spark.createDataFrame(payroll_data, schema=payroll_schema)


# ==============================================================================
# 4. SOLUTION FUNCTION
# ==============================================================================
# def compute_payroll(df_emp, df_pay):
#     """
#     Computes weekly paychecks with overtime using PySpark DataFrame API.
#     """
#     joined_df = df_pay.join(df_emp, on="employee_id", how="inner")

#     pay_expression = F.when(
#         F.col("hours_worked") > 40.0,
#         (40.0 * F.col("hourly_rate")) +
#         ((F.col("hours_worked") - 40.0) * F.col("hourly_rate") * 1.5)
#     ).otherwise(
#         F.col("hours_worked") * F.col("hourly_rate")
#     )

#     result_df = joined_df.withColumn("pay", F.round(pay_expression, 2)) \
#                          .select("employee_id", "name", "pay", "position") \
#                          .orderBy(F.col("employee_id").asc())

#     return result_df


# # Run solution on Example 1
# result_df = compute_payroll(df_employees, df_payroll)
# print("--- Result DataFrame ---")
# result_df.show(truncate=False)

In [4]:
df_employees.show()

+-----------+-----+---+-----------------+
|employee_id| name|age|         position|
+-----------+-----+---+-----------------+
|          1|Alice| 25|Software Engineer|
|          2|  Bob| 30|     Data Analyst|
|          3|Carol| 28|  Product Manager|
|          4| Dave| 24|Software Engineer|
|          5|  Eve| 31|      QA Engineer|
+-----------+-----+---+-----------------+



In [5]:
df_payroll.show()

+-----------+------------+-----------+
|employee_id|hours_worked|hourly_rate|
+-----------+------------+-----------+
|          1|        45.0|       30.0|
|          2|        38.0|       25.0|
|          3|        41.5|       35.0|
|          4|        40.0|       28.0|
+-----------+------------+-----------+



In [13]:
# Building Code
joined_df = df_employees.join(df_payroll, on = 'employee_id', how = 'inner').select(
        F.col('employee_id'),
        F.col('name'),
        F.col('hours_worked'),
        F.col('hourly_rate'),
        F.when(F.col('hours_worked') > 40, F.col('hours_worked') - F.lit(40)).alias('overtime_hrs'),
        F.col('position')
    )

trans_df = joined_df.withColumn('pay', F.when(F.col('overtime_hrs').isNotNull(), \
            (F.lit(40) * F.col('hourly_rate')) + \
            (F.col('overtime_hrs') * (F.col('hourly_rate') * F.lit(1.5)))) \
            .otherwise((F.col('hours_worked') * F.col('hourly_rate'))))

result = trans_df.select(
    F.col('employee_id'),
    F.col('name'),
    F.round(F.col('pay'), 2).alias('pay'),
    F.col('position')
).orderBy(F.col('employee_id').asc())

result.show()


+-----------+-----+-------+-----------------+
|employee_id| name|    pay|         position|
+-----------+-----+-------+-----------------+
|          1|Alice| 1425.0|Software Engineer|
|          2|  Bob|  950.0|     Data Analyst|
|          3|Carol|1478.75|  Product Manager|
|          4| Dave| 1120.0|Software Engineer|
+-----------+-----+-------+-----------------+



In [14]:
# ==============================================================================
# 4. SOLUTION FUNCTION
# ==============================================================================

def compute_payroll(df_emp, df_pay):
  joined_df = df_emp.join(df_pay, on = 'employee_id', how = 'inner').select(
        F.col('employee_id'),
        F.col('name'),
        F.col('hours_worked'),
        F.col('hourly_rate'),
        F.when(F.col('hours_worked') > 40, F.col('hours_worked') - F.lit(40)).alias('overtime_hrs'),
        F.col('position')
    )

  trans_df = joined_df.withColumn('pay', F.when(F.col('overtime_hrs').isNotNull(), \
              (F.lit(40) * F.col('hourly_rate')) + \
              (F.col('overtime_hrs') * (F.col('hourly_rate') * F.lit(1.5)))) \
              .otherwise((F.col('hours_worked') * F.col('hourly_rate'))))

  result = trans_df.select(
      F.col('employee_id'),
      F.col('name'),
      F.round(F.col('pay'), 2).alias('pay'),
      F.col('position')
  ).orderBy(F.col('employee_id').asc())

  return result

In [15]:
# ==============================================================================
# 5. TEST CASES / ASSERTIONS
# ==============================================================================
def run_tests():
    # Test Case 1: Example 1 Verification
    expected_rows_tc1 = [
        (1, "Alice", 1425.00, "Software Engineer"),
        (2, "Bob", 950.00, "Data Analyst"),
        (3, "Carol", 1478.75, "Product Manager"),
        (4, "Dave", 1120.00, "Software Engineer")
    ]

    result_df = compute_payroll(df_employees, df_payroll)

    actual_rows_tc1 = [
        (r.employee_id, r.name, float(r.pay), r.position)
        for r in result_df.collect()
    ]

    assert actual_rows_tc1 == expected_rows_tc1, f"TC1 Failed! Got: {actual_rows_tc1}"
    print("✅ [PASSED] Test Case 1: Example 1 matches expected output.")

    # Test Case 2: Boundary & Edge Cases (Zero hours, fractional overtime, high hours)
    tc2_emp = spark.createDataFrame([
        (10, "Intern Zero", 20, "Intern"),
        (20, "Precise OT", 26, "Designer"),
        (30, "Overtime Champ", 38, "Site Reliability Engineer")
    ], schema=emp_schema)

    tc2_pay = spark.createDataFrame([
        (10, 0.0, 18.50),    # 0 hrs -> 0.00
        (20, 40.25, 40.00),  # 40 * 40 + (0.25 * 40 * 1.5) = 1600 + 15 = 1615.00
        (30, 60.0, 50.00)    # 40 * 50 + (20 * 50 * 1.5) = 2000 + 1500 = 3500.00
    ], schema=payroll_schema)

    expected_rows_tc2 = [
        (10, "Intern Zero", 0.00, "Intern"),
        (20, "Precise OT", 1615.00, "Designer"),
        (30, "Overtime Champ", 3500.00, "Site Reliability Engineer")
    ]

    result_tc2 = compute_payroll(tc2_emp, tc2_pay)
    actual_rows_tc2 = [
        (r.employee_id, r.name, float(r.pay), r.position)
        for r in result_tc2.collect()
    ]

    assert actual_rows_tc2 == expected_rows_tc2, f"TC2 Failed! Got: {actual_rows_tc2}"
    print("✅ [PASSED] Test Case 2: Zero hours, small fractional overtime, and high overtime.")

run_tests()

✅ [PASSED] Test Case 1: Example 1 matches expected output.
✅ [PASSED] Test Case 2: Zero hours, small fractional overtime, and high overtime.
